# Individual Macaque Identification via Clustering

This notebook uses clustering techniques to identify individual macaques from their visual features:

1. Load tracklet features from detection notebook
2. Apply dimensionality reduction (UMAP/PCA)
3. Cluster tracklets using HDBSCAN
4. Validate and refine clustering results
5. Create individual identity mappings


In [ ]:
import sys
import os
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import umap
import warnings
warnings.filterwarnings('ignore')

from macaque_tracker.clustering import IndividualIdentifier

plt.style.use('default')
sns.set_palette("husl")

## 1. Load Detection Results and Features

In [ ]:
# Find available result files
output_dir = Path("../output")

detection_files = list(output_dir.glob("detections_*.csv"))
feature_files = list(output_dir.glob("tracklet_features_*.npz"))
mapping_files = list(output_dir.glob("track_mapping_*.csv"))

print(f"Found {len(detection_files)} detection files")
print(f"Found {len(feature_files)} feature files")
print(f"Found {len(mapping_files)} mapping files")

if detection_files and feature_files:
    # Use the first available set of results
    detection_file = detection_files[0]
    feature_file = feature_files[0]
    
    print(f"\nUsing:")
    print(f"Detections: {detection_file.name}")
    print(f"Features: {feature_file.name}")
else:
    print("\n⚠️  No detection results found. Please run the detection notebook first.")

In [ ]:
# Load detection results
if detection_files and feature_files:
    df_detections = pd.read_csv(detection_file)
    
    # Load tracklet features
    features_data = np.load(feature_file)
    
    # Reconstruct tracklet features dictionary
    tracklet_features = {}
    for key in features_data.files:
        track_id = int(key.replace('track_', ''))
        tracklet_features[track_id] = features_data[key]
    
    print(f"Loaded {len(df_detections)} detections")
    print(f"Loaded features for {len(tracklet_features)} tracklets")
    print(f"Feature dimension: {list(tracklet_features.values())[0].shape[0] if tracklet_features else 'N/A'}")
    
    # Basic statistics
    print(f"\nTracklet Statistics:")
    track_lengths = df_detections.groupby('track_id').size()
    print(f"Number of tracklets: {len(track_lengths)}")
    print(f"Average tracklet length: {track_lengths.mean():.1f} detections")
    print(f"Tracklet length range: {track_lengths.min()} - {track_lengths.max()}")

## 2. Feature Analysis and Preprocessing

In [ ]:
if tracklet_features:
    # Filter tracklets with sufficient features
    valid_features = {k: v for k, v in tracklet_features.items() 
                     if len(v) > 0 and not np.all(v == 0)}
    
    print(f"Valid tracklets with features: {len(valid_features)} / {len(tracklet_features)}")
    
    if len(valid_features) >= 3:  # Need at least 3 tracklets for clustering
        # Prepare feature matrix
        track_ids = list(valid_features.keys())
        feature_matrix = np.array(list(valid_features.values()))
        
        print(f"Feature matrix shape: {feature_matrix.shape}")
        print(f"Feature statistics:")
        print(f"  Mean: {feature_matrix.mean():.3f}")
        print(f"  Std: {feature_matrix.std():.3f}")
        print(f"  Min: {feature_matrix.min():.3f}")
        print(f"  Max: {feature_matrix.max():.3f}")
        
        # Check for any problematic features
        zero_variance_features = np.var(feature_matrix, axis=0) == 0
        print(f"Features with zero variance: {zero_variance_features.sum()}")
        
        # Remove zero variance features if any
        if zero_variance_features.sum() > 0:
            feature_matrix = feature_matrix[:, ~zero_variance_features]
            print(f"Cleaned feature matrix shape: {feature_matrix.shape}")
    else:
        print("⚠️  Not enough valid tracklets for clustering")
else:
    print("⚠️  No tracklet features available")

In [ ]:
# Visualize feature distributions
if 'feature_matrix' in locals() and len(valid_features) >= 3:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Feature value distribution
    axes[0,0].hist(feature_matrix.flatten(), bins=50, alpha=0.7)
    axes[0,0].set_title('Distribution of All Feature Values')
    axes[0,0].set_xlabel('Feature Value')
    axes[0,0].set_ylabel('Frequency')
    
    # Feature variance across tracklets
    feature_vars = np.var(feature_matrix, axis=0)
    axes[0,1].hist(feature_vars, bins=30, alpha=0.7)
    axes[0,1].set_title('Feature Variance Distribution')
    axes[0,1].set_xlabel('Variance')
    axes[0,1].set_ylabel('Number of Features')
    
    # Sample of feature vectors (first 10 features, first 20 tracklets)
    sample_features = feature_matrix[:min(20, len(track_ids)), :min(10, feature_matrix.shape[1])]
    im = axes[1,0].imshow(sample_features, aspect='auto', cmap='viridis')
    axes[1,0].set_title('Sample Feature Matrix (Tracklets × Features)')
    axes[1,0].set_xlabel('Feature Index')
    axes[1,0].set_ylabel('Tracklet Index')
    plt.colorbar(im, ax=axes[1,0])
    
    # Correlation between some features
    if feature_matrix.shape[1] >= 2:
        axes[1,1].scatter(feature_matrix[:, 0], feature_matrix[:, 1], alpha=0.6)
        axes[1,1].set_title('Feature 0 vs Feature 1')
        axes[1,1].set_xlabel('Feature 0')
        axes[1,1].set_ylabel('Feature 1')
    else:
        axes[1,1].text(0.5, 0.5, 'Not enough features\nfor scatter plot', 
                      ha='center', va='center', transform=axes[1,1].transAxes)
    
    plt.tight_layout()
    plt.show()

## 3. Dimensionality Reduction and Visualization

In [ ]:
# Apply dimensionality reduction for visualization
if 'feature_matrix' in locals() and len(valid_features) >= 3:
    # Standardize features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(feature_matrix)
    
    # Apply PCA
    n_components_pca = min(10, len(track_ids)-1, features_scaled.shape[1])
    if n_components_pca >= 2:
        pca = PCA(n_components=n_components_pca)
        features_pca = pca.fit_transform(features_scaled)
        
        print(f"PCA Results:")
        print(f"Explained variance ratio (first 5 components): {pca.explained_variance_ratio_[:5]}")
        print(f"Cumulative explained variance: {pca.explained_variance_ratio_.cumsum()[:5]}")
    
    # Apply UMAP if we have enough samples
    if len(track_ids) >= 4:
        n_neighbors = min(15, len(track_ids)//2)
        umap_reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors, random_state=42)
        features_umap = umap_reducer.fit_transform(features_scaled)
        print(f"\nUMAP completed with n_neighbors={n_neighbors}")
    
    # Apply t-SNE if we have enough samples  
    if len(track_ids) >= 4:
        perplexity = min(30, (len(track_ids)-1)//3)
        tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
        features_tsne = tsne.fit_transform(features_scaled)
        print(f"t-SNE completed with perplexity={perplexity}")

In [ ]:
# Visualize dimensionality reduction results
if 'features_pca' in locals() or 'features_umap' in locals() or 'features_tsne' in locals():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # PCA plot
    if 'features_pca' in locals() and features_pca.shape[1] >= 2:
        scatter = axes[0].scatter(features_pca[:, 0], features_pca[:, 1], 
                                 c=range(len(track_ids)), cmap='tab20', s=50)
        axes[0].set_title(f'PCA Projection\n({pca.explained_variance_ratio_[:2].sum():.1%} variance explained)')
        axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
        axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
        
        # Add track ID annotations for first few points
        for i in range(min(10, len(track_ids))):
            axes[0].annotate(f'T{track_ids[i]}', (features_pca[i, 0], features_pca[i, 1]), 
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
    else:
        axes[0].text(0.5, 0.5, 'PCA not available', ha='center', va='center')
        axes[0].set_title('PCA Projection')
    
    # UMAP plot
    if 'features_umap' in locals():
        scatter = axes[1].scatter(features_umap[:, 0], features_umap[:, 1], 
                                 c=range(len(track_ids)), cmap='tab20', s=50)
        axes[1].set_title('UMAP Projection')
        axes[1].set_xlabel('UMAP 1')
        axes[1].set_ylabel('UMAP 2')
        
        # Add track ID annotations for first few points
        for i in range(min(10, len(track_ids))):
            axes[1].annotate(f'T{track_ids[i]}', (features_umap[i, 0], features_umap[i, 1]), 
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
    else:
        axes[1].text(0.5, 0.5, 'UMAP not available', ha='center', va='center')
        axes[1].set_title('UMAP Projection')
    
    # t-SNE plot
    if 'features_tsne' in locals():
        scatter = axes[2].scatter(features_tsne[:, 0], features_tsne[:, 1], 
                                 c=range(len(track_ids)), cmap='tab20', s=50)
        axes[2].set_title('t-SNE Projection')
        axes[2].set_xlabel('t-SNE 1')
        axes[2].set_ylabel('t-SNE 2')
        
        # Add track ID annotations for first few points
        for i in range(min(10, len(track_ids))):
            axes[2].annotate(f'T{track_ids[i]}', (features_tsne[i, 0], features_tsne[i, 1]), 
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
    else:
        axes[2].text(0.5, 0.5, 't-SNE not available', ha='center', va='center')
        axes[2].set_title('t-SNE Projection')
    
    plt.tight_layout()
    plt.show()

## 4. Clustering Analysis

In [ ]:
# Initialize and run clustering
if 'valid_features' in locals() and len(valid_features) >= 3:
    # Try different clustering parameters
    clustering_results = {}
    
    # HDBSCAN with different parameters
    min_cluster_sizes = [2, 3, max(2, len(valid_features)//4)]
    
    for min_size in min_cluster_sizes:
        if min_size <= len(valid_features)//2:  # Ensure reasonable cluster size
            try:
                identifier = IndividualIdentifier(method='hdbscan', 
                                                min_cluster_size=min_size,
                                                min_samples=max(1, min_size-1))
                track_to_individual = identifier.fit_predict(valid_features)
                
                # Calculate clustering metrics
                labels = list(track_to_individual.values())
                n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
                n_noise = list(labels).count(-1)
                
                if n_clusters > 0 and len(set(labels)) > 1:
                    silhouette = silhouette_score(identifier.features_, labels) if len(set(labels)) > 1 else -1
                    calinski = calinski_harabasz_score(identifier.features_, labels) if len(set(labels)) > 1 else 0
                else:
                    silhouette = -1
                    calinski = 0
                
                clustering_results[f'hdbscan_minsize_{min_size}'] = {
                    'identifier': identifier,
                    'mapping': track_to_individual,
                    'n_clusters': n_clusters,
                    'n_noise': n_noise,
                    'silhouette': silhouette,
                    'calinski': calinski
                }
                
                print(f"HDBSCAN (min_size={min_size}): {n_clusters} individuals, {n_noise} noise, silhouette={silhouette:.3f}")
                
            except Exception as e:
                print(f"HDBSCAN (min_size={min_size}) failed: {e}")
    
    # DBSCAN with different parameters
    eps_values = [0.3, 0.5, 1.0]
    for eps in eps_values:
        try:
            identifier = IndividualIdentifier(method='dbscan', eps=eps, min_samples=2)
            track_to_individual = identifier.fit_predict(valid_features)
            
            labels = list(track_to_individual.values())
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
            n_noise = list(labels).count(-1)
            
            if n_clusters > 0 and len(set(labels)) > 1:
                silhouette = silhouette_score(identifier.features_, labels)
                calinski = calinski_harabasz_score(identifier.features_, labels)
            else:
                silhouette = -1
                calinski = 0
            
            clustering_results[f'dbscan_eps_{eps}'] = {
                'identifier': identifier,
                'mapping': track_to_individual,
                'n_clusters': n_clusters,
                'n_noise': n_noise,
                'silhouette': silhouette,
                'calinski': calinski
            }
            
            print(f"DBSCAN (eps={eps}): {n_clusters} individuals, {n_noise} noise, silhouette={silhouette:.3f}")
            
        except Exception as e:
            print(f"DBSCAN (eps={eps}) failed: {e}")
            
    print(f"\nTested {len(clustering_results)} clustering configurations")

In [ ]:
# Compare clustering results
if clustering_results:
    results_df = pd.DataFrame([
        {
            'method': name,
            'n_clusters': result['n_clusters'],
            'n_noise': result['n_noise'],
            'silhouette': result['silhouette'],
            'calinski': result['calinski'],
            'coverage': (len(valid_features) - result['n_noise']) / len(valid_features)
        }
        for name, result in clustering_results.items()
    ])
    
    print("Clustering Results Comparison:")
    display(results_df.round(3))
    
    # Select best clustering based on silhouette score and reasonable number of clusters
    valid_results = results_df[(results_df['n_clusters'] > 0) & (results_df['n_clusters'] <= len(valid_features)//2)]
    
    if not valid_results.empty:
        best_method = valid_results.loc[valid_results['silhouette'].idxmax(), 'method']
        best_result = clustering_results[best_method]
        print(f"\nBest clustering method: {best_method}")
        print(f"Number of proposed individuals: {best_result['n_clusters']}")
        print(f"Tracklets assigned to noise: {best_result['n_noise']}")
        print(f"Silhouette score: {best_result['silhouette']:.3f}")
    else:
        print("\nNo valid clustering results found")
        best_method = list(clustering_results.keys())[0]  # Use first available
        best_result = clustering_results[best_method]
        print(f"Using first available method: {best_method}")

## 5. Visualize Best Clustering Results

In [ ]:
# Visualize the best clustering result
if 'best_result' in locals():
    identifier = best_result['identifier']
    track_to_individual = best_result['mapping']
    
    # Create visualization
    fig = identifier.visualize_clusters()
    plt.suptitle(f'Individual Identification Results\n({best_method})', fontsize=14, y=1.02)
    plt.show()
    
    # Show cluster statistics
    cluster_stats = identifier.get_cluster_statistics()
    print("\nCluster Statistics:")
    display(cluster_stats)

In [ ]:
# Interactive visualization with plotly if dimensionality reduction is available
if 'features_umap' in locals() and 'best_result' in locals():
    # Prepare data for plotting
    plot_data = pd.DataFrame({
        'UMAP_1': features_umap[:, 0],
        'UMAP_2': features_umap[:, 1],
        'Track_ID': track_ids,
        'Individual_ID': [track_to_individual.get(tid, -1) for tid in track_ids]
    })
    
    # Create interactive plot
    fig = px.scatter(plot_data, x='UMAP_1', y='UMAP_2', 
                     color='Individual_ID', 
                     hover_data=['Track_ID'],
                     title=f'Interactive Individual Clusters ({best_method})',
                     color_continuous_scale='tab20')
    
    fig.update_traces(marker=dict(size=8, line=dict(width=1, color='black')))
    fig.update_layout(width=800, height=600)
    fig.show()
    
elif 'features_pca' in locals() and 'best_result' in locals() and features_pca.shape[1] >= 2:
    # Fallback to PCA if UMAP not available
    plot_data = pd.DataFrame({
        'PC_1': features_pca[:, 0],
        'PC_2': features_pca[:, 1], 
        'Track_ID': track_ids,
        'Individual_ID': [track_to_individual.get(tid, -1) for tid in track_ids]
    })
    
    fig = px.scatter(plot_data, x='PC_1', y='PC_2', 
                     color='Individual_ID',
                     hover_data=['Track_ID'],
                     title=f'Interactive Individual Clusters - PCA ({best_method})',
                     color_continuous_scale='tab20')
    
    fig.update_traces(marker=dict(size=8, line=dict(width=1, color='black')))
    fig.update_layout(width=800, height=600)
    fig.show()

## 6. Individual Profile Analysis

In [ ]:
# Analyze individual profiles
if 'best_result' in locals() and 'df_detections' in locals():
    # Add individual IDs to detection dataframe
    df_detections['individual_id'] = df_detections['track_id'].map(track_to_individual)
    
    # Individual statistics
    individual_stats = df_detections.groupby('individual_id').agg({
        'track_id': 'nunique',
        'frame': ['count', 'min', 'max'],
        'confidence': ['mean', 'std'],
        'bbox_x1': 'mean',
        'bbox_y1': 'mean',
        'center_x': 'mean',
        'center_y': 'mean'
    }).round(2)
    
    individual_stats.columns = ['num_tracks', 'total_detections', 'first_frame', 'last_frame',
                               'avg_confidence', 'std_confidence', 'avg_x1', 'avg_y1', 'avg_center_x', 'avg_center_y']
    
    # Calculate temporal span
    individual_stats['frame_span'] = individual_stats['last_frame'] - individual_stats['first_frame']
    
    print("Individual Macaque Profiles:")
    display(individual_stats)
    
    # Show only non-noise individuals
    valid_individuals = individual_stats[individual_stats.index >= 0]
    if not valid_individuals.empty:
        print(f"\nSummary of {len(valid_individuals)} proposed individuals:")
        print(f"Total tracklets assigned: {valid_individuals['num_tracks'].sum()}")
        print(f"Total detections: {valid_individuals['total_detections'].sum()}")
        print(f"Average detections per individual: {valid_individuals['total_detections'].mean():.1f}")
        print(f"Average tracklets per individual: {valid_individuals['num_tracks'].mean():.1f}")


In [ ]:
# Visualize individual temporal and spatial patterns
if 'valid_individuals' in locals() and not valid_individuals.empty:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Detection count per individual
    valid_individuals['total_detections'].plot(kind='bar', ax=axes[0,0])
    axes[0,0].set_title('Detections per Proposed Individual')
    axes[0,0].set_xlabel('Individual ID')
    axes[0,0].set_ylabel('Number of Detections')
    axes[0,0].tick_params(axis='x', rotation=0)
    
    # Temporal span per individual
    valid_individuals['frame_span'].plot(kind='bar', ax=axes[0,1])
    axes[0,1].set_title('Temporal Span per Individual')
    axes[0,1].set_xlabel('Individual ID')
    axes[0,1].set_ylabel('Frame Span')
    axes[0,1].tick_params(axis='x', rotation=0)
    
    # Spatial distribution of individuals
    colors = plt.cm.tab20(np.linspace(0, 1, len(valid_individuals)))
    for i, (individual_id, color) in enumerate(zip(valid_individuals.index, colors)):
        individual_detections = df_detections[df_detections['individual_id'] == individual_id]
        axes[1,0].scatter(individual_detections['center_x'], individual_detections['center_y'], 
                         c=[color], label=f'Individual {individual_id}', alpha=0.6, s=20)
    
    axes[1,0].set_title('Spatial Distribution by Individual')
    axes[1,0].set_xlabel('Center X')
    axes[1,0].set_ylabel('Center Y')
    axes[1,0].invert_yaxis()
    axes[1,0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Confidence distribution per individual
    individual_confidences = []
    for individual_id in valid_individuals.index:
        individual_detections = df_detections[df_detections['individual_id'] == individual_id]
        individual_confidences.append(individual_detections['confidence'].values)
    
    axes[1,1].boxplot(individual_confidences, labels=valid_individuals.index)
    axes[1,1].set_title('Detection Confidence by Individual')
    axes[1,1].set_xlabel('Individual ID')
    axes[1,1].set_ylabel('Detection Confidence')
    
    plt.tight_layout()
    plt.show()

## 7. Save Results

In [ ]:
# Save clustering results
if 'best_result' in locals() and 'df_detections' in locals():
    # Save individual mapping
    individual_mapping_df = pd.DataFrame([
        {'track_id': track_id, 'individual_id': individual_id}
        for track_id, individual_id in track_to_individual.items()
    ])
    
    mapping_file = output_dir / f"individual_mapping_{detection_file.stem.replace('detections_', '')}.csv"
    individual_mapping_df.to_csv(mapping_file, index=False)
    print(f"✓ Saved individual mapping to {mapping_file}")
    
    # Save enhanced detection results with individual IDs
    enhanced_detections_file = output_dir / f"detections_with_individuals_{detection_file.stem.replace('detections_', '')}.csv"
    df_detections.to_csv(enhanced_detections_file, index=False)
    print(f"✓ Saved enhanced detections to {enhanced_detections_file}")
    
    # Save individual statistics
    if 'individual_stats' in locals():
        individual_stats_file = output_dir / f"individual_statistics_{detection_file.stem.replace('detections_', '')}.csv"
        individual_stats.to_csv(individual_stats_file)
        print(f"✓ Saved individual statistics to {individual_stats_file}")
    
    # Save clustering parameters and results
    clustering_summary = {
        'method': best_method,
        'n_individuals': best_result['n_clusters'],
        'n_noise_tracklets': best_result['n_noise'],
        'silhouette_score': best_result['silhouette'],
        'calinski_harabasz_score': best_result['calinski'],
        'total_tracklets': len(valid_features),
        'coverage_ratio': (len(valid_features) - best_result['n_noise']) / len(valid_features)
    }
    
    summary_df = pd.DataFrame([clustering_summary])
    summary_file = output_dir / f"clustering_summary_{detection_file.stem.replace('detections_', '')}.csv"
    summary_df.to_csv(summary_file, index=False)
    print(f"✓ Saved clustering summary to {summary_file}")

## Summary

This notebook analyzed tracklet features and identified individual macaques:

### Results:
- ✅ Loaded tracklet features from detection phase
- ✅ Applied dimensionality reduction (PCA, UMAP, t-SNE)
- ✅ Tested multiple clustering algorithms (HDBSCAN, DBSCAN)
- ✅ Selected best clustering based on silhouette score
- ✅ Generated individual identity mappings
- ✅ Analyzed individual profiles and characteristics

### Files Created:
- `../output/individual_mapping_*.csv` - Track ID to Individual ID mapping
- `../output/detections_with_individuals_*.csv` - Enhanced detections with individual IDs
- `../output/individual_statistics_*.csv` - Statistics for each proposed individual
- `../output/clustering_summary_*.csv` - Clustering method and performance metrics

### Next Steps:
- Use the video extraction notebook to create clips for each individual
- Review and validate the proposed individual identifications
- Consider manual refinement if needed